In [3]:
import polars as pl
version = pl.__version__
print(f"Polars version: {version}")

Polars version: 1.34.0


In [8]:
import requests
import json
import numpy as np
import time
from tqdm import tqdm
import datetime
from constant import data_path
base_path = f"{data_path}/medrxiv"

In [5]:
import logging
import re
from dateutil.parser import parse as parse_date

logging.basicConfig(
    filename='log/medrxiv_log.log',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
logging.info("Starting the script")

def clean_whitespace(d):
    def clean_string(s):
        # Replace any whitespace character (except normal space) with a normal space
        s = ''.join(c if ord(c) == 32 or not c.isspace() else ' ' for c in s)
        # Collapse multiple spaces and strip leading/trailing
        return re.sub(r'\s{2,}', ' ', s).strip()
    
    return {k: clean_string(v) if isinstance(v, str) else v for k, v in d.items()}
    
def normalize(article):
    article = clean_whitespace(article)
    if "authors" in article.keys(): article["authors"] = [author for author in article["authors"].split(";") if author.strip() != ""]
    if "preprint_authors" in article.keys(): article["preprint_authors"] = [author for author in article["preprint_authors"].split(";") if author.strip() != ""]
    if "category" in article.keys(): article["category"] = article["category"].strip()
    if "version" in article.keys(): article["version"] = int(article["version"])
    if "published" in article.keys(): article["published"] = article["published"] if article["published"]!="NA" else None
    return article

def fetch_articles(url):
    res = requests.get(url + "0")
    r = json.loads(res.text)

    if len(r["collection"])==0:
        logging.info(r["messages"])
        return
    
    total = int(r["messages"][0]["total"])
    if total == 0:
        logging.info("No articles found")
        return
    logging.info(f"Total articles found: {total}")

    for cursor in tqdm(range(int(np.ceil(total/100)))):
        try:
            res = requests.get(url + f"{cursor * 100}")
            logging.info(f"Fetching batch {cursor}")
            r = res.json()
            if len(r["collection"]) == 0:
                logging.info("No more articles found")
                break
            articles = [json.dumps(normalize(a)) for a in r["collection"]]
            yield articles
            time.sleep(1)
        except Exception as e:
            logging.error(f"Error fetching batch {cursor}: {e}")
    return 

def monthrange(any_day):
    next_month = any_day.replace(day=28) + datetime.timedelta(days=4)
    last_day_of_month = next_month - datetime.timedelta(days=next_month.day)
    first_day_of_month = any_day.replace(day=1)

    return first_day_of_month.strftime('%Y-%m-%d'), last_day_of_month.strftime('%Y-%m-%d')

In [23]:
year = 2019
url = f"https://api.medrxiv.org/details/medrxiv/{year}-01-01/{year}-12-31/"
res = requests.get(url + "0")
r = json.loads(res.text)
r

{'messages': [{'status': 'ok',
   'interval': '2019-01-01:2019-12-31',
   'cursor': 0,
   'count': 100,
   'count_new_papers': 913,
   'total': 1055}],
 'collection': [{'doi': '10.1101/19000109',
   'title': 'Molecular profiling of neonatal dried blood spots reveals changes in innate and adaptive immunity following fetal inflammatory response',
   'authors': 'Daniel Costa; Nuria Bonet; Amanda Sole; Jose Manuel Gonzalez de Aledo-Castillo; Eduard Sabido; Ferran Casals; Carlota Rovira; Alfons Nadal; Jose Luis Marin; Teresa Cobo; Robert Castelo',
   'author_corresponding': 'Robert Castelo',
   'author_corresponding_institution': 'Department of Experimental and Health Sciences, Universitat Pompeu Fabra (UPF), Barcelona, Spain; Research Programme on Biomedical Informatics, Institut Hospita',
   'date': '2019-06-25',
   'version': '1',
   'type': '',
   'license': 'cc_by',
   'category': 'genetic and genomic medicine',
   'jatsxml': 'https://www.medrxiv.org/content/early/2019/06/25/19000109.s

In [ ]:
BATCH_SIZE = 1000
import os

for year in range(2019, 2025):
    start, end = f"{year}-01-01", f"{year}-12-31"
    buffer = []
    url = f"https://api.medrxiv.org/details/medrxiv/{start}/{end}/"

    path = f"{base_path}/full_{year}.jsonl"
    # Check if the file already exists
    if os.path.exists(path):
        logging.info(f"File {path} already exists, skipping...")
        continue
    else:
        logging.info(f"Creating file {path}...")
        os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        for article in fetch_articles(url):
            buffer.extend(article)
            if len(buffer) >= BATCH_SIZE:
                f.write("\n".join(buffer) + "\n")
                buffer.clear()

        if buffer:
            f.write("\n".join(buffer) + "\n")

In [33]:
BATCH_SIZE = 1000

import os

for year in range(2019, 2025):
    start, end = f"{year}-01-01", f"{year}-12-31"
    buffer = []
    url = f"https://api.biorxiv.org/pubs/medrxiv/{start}/{end}/"
    
    path = f"{base_path}/curated_detail_{year}.jsonl"
    # Check if the file already exists
    if os.path.exists(path):
        logging.info(f"File {path} already exists, skipping...")
        continue

    with open(path, "a", encoding="utf-8") as f:
        for article in fetch_articles(url):
            buffer.extend(article)
            if len(buffer) >= BATCH_SIZE:
                f.write("\n".join(buffer) + "\n")
                buffer.clear()

        if buffer:
            f.write("\n".join(buffer) + "\n")

100%|████████████████████████████████████████████████████████████| 46/46 [01:28<00:00,  1.92s/it]


# export all dois and curated dois

In [9]:
import polars as pl
# concat all jsonl files vertically
df = pl.concat([
    pl.read_ndjson(f"{base_path}/full_{year}.jsonl")
    for year in range(2019, 2025)
]) 
df.head()

doi,title,authors,author_corresponding,author_corresponding_institution,date,version,type,license,category,jatsxml,abstract,published,server
str,str,list[str],str,str,str,i64,str,str,str,str,str,str,str
"""10.1101/19000109""","""Molecular profiling of neonata…","[""Daniel Costa"", "" Nuria Bonet"", … "" Robert Castelo""]","""Robert Castelo""","""Department of Experimental and…","""2019-06-25""",1,"""""","""cc_by""","""genetic and genomic medicine""","""https://www.medrxiv.org/conten…","""The fetal inflammatory respons…","""10.1111/febs.15578""","""medrxiv"""
"""10.1101/19000273""","""Crohns disease and ulcerative …","[""Orna G Ehrlich"", "" James Testaverde"", … "" Peter D.R. Higgins""]","""Orna G Ehrlich""","""Crohn\'s & Colitis Foundation""","""2019-06-25""",1,"""""","""cc_by_nc_nd""","""gastroenterology""","""https://www.medrxiv.org/conten…","""BackgroundClinical trial recru…",null,"""medrxiv"""
"""10.1101/19000463""","""Updating Insights into Rosigli…","[""Joshua D Wallach"", "" Kun Wang"", … "" Joseph S. Ross""]","""Joshua D Wallach""","""Yale School of Public Health""","""2019-06-25""",1,"""""","""cc_by_nc""","""cardiovascular medicine""","""https://www.medrxiv.org/conten…","""ObjectiveTo conduct a systemat…","""10.1136/bmj.l7078""","""medrxiv"""
"""10.1101/19000430""","""Predicting epileptic seizures …","[""Olivera Stojanovic"", "" Gordon Pipa""]","""Olivera Stojanovic""","""Institute of Cognitive Science…","""2019-06-25""",1,"""""","""cc_by""","""health informatics""","""https://www.medrxiv.org/conten…","""This paper presents a procedur…","""10.1371/journal.pone.0228025""","""medrxiv"""
"""10.1101/19000133""","""Prospective and External Evalu…","[""Nathan Brajer"", "" Brian Cozzi"", … "" Mark Sendak""]","""Nathan Brajer""","""Duke University School of Medi…","""2019-06-25""",1,"""""","""cc_by_nc_nd""","""health systems and quality imp…","""https://www.medrxiv.org/conten…","""The ability to accurately pred…","""10.1001/jamanetworkopen.2019.2…","""medrxiv"""


In [10]:
df_unique = df.unique("doi")

duplicate_published_dois = df_unique.filter(
    pl.col("published").is_not_null(),
    pl.col("published")!="",
    pl.col("published").is_duplicated()
).select("doi").to_series().unique().to_list()

print(df.height, df_unique.height)
print(len(duplicate_published_dois))
df.filter(
    pl.col("doi").is_null()
)

77710 62166
147


doi,title,authors,author_corresponding,author_corresponding_institution,date,version,type,license,category,jatsxml,abstract,published,server
str,str,list[str],str,str,str,i64,str,str,str,str,str,str,str


In [ ]:
df_cleaned = df_unique.filter(
    ~pl.col("doi").is_in(duplicate_published_dois)
)

doi,title,authors,author_corresponding,author_corresponding_institution,date,version,type,license,category,jatsxml,abstract,published,server
str,str,list[str],str,str,str,i64,str,str,str,str,str,str,str
"""10.1101/2020.07.19.20157198""","""High Community SARS-CoV-2 Anti…","[""McLaughlin, C."", "" Doll, M. K."", … "" Corey, L.""]","""Colleen McLaughlin""","""Albany College of Pharmacy and…","""2020-07-21""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""infectious diseases""","""https://www.medrxiv.org/conten…","""Community-level seroprevalence…",null,"""medrxiv"""
"""10.1101/2020.09.01.20186213""","""Performance of serum apolipopr…","[""Poynard, T."", "" Deckmyn, O."", … "" Cacoub, P.""]","""Thierry Poynard""","""APHP Sorbonne University""","""2020-09-03""",1,"""PUBLISHAHEADOFPRINT""","""cc_no""","""infectious diseases""","""https://www.medrxiv.org/conten…","""BackgroundSince 1920, a decrea…","""10.1371/journal.pone.0242306""","""medrxiv"""
"""10.1101/2023.06.26.23291780""","""Giving birth is like going to …","[""Alio, A. P."", "" Garba, R. M."", … "" Moha, M.""]","""Amina Paula Alio""","""University of Rochester""","""2023-06-29""",1,"""PUBLISHAHEADOFPRINT""","""cc_no""","""obstetrics and gynecology""","""https://www.medrxiv.org/conten…","""Obstetric violence has been re…",null,"""medrxiv"""
"""10.1101/2024.09.26.24314455""","""Detection of diphtheria toxin-…","[""Lowe, C. F."", "" Ritchie, G."", … "" Romney, M. G.""]","""Christopher F. Lowe""","""St. Paul\'s Hospital""","""2024-09-27""",1,"""PUBLISHAHEADOFPRINT""","""cc_by""","""infectious diseases""","""https://www.medrxiv.org/conten…","""We conducted a retrospective c…","""10.3201/eid3102.241472""","""medrxiv"""
"""10.1101/2021.05.26.21257872""","""Cohort Design and Natural Lang…","[""Khurshid, S."", "" Reeder, C."", … "" Lubitz, S.""]","""Steven Lubitz""","""Massachusetts General Hospital""","""2021-05-30""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""epidemiology""","""https://www.medrxiv.org/conten…","""BackgroundElectronic health re…","""10.1038/s41746-022-00590-0""","""medrxiv"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""10.1101/2022.01.18.22269200""","""Infection and transmission ris…","[""Heinsohn, T."", "" Lange, B."", … "" Krause, G.""]","""Torben Heinsohn""","""Helmholtz Centre for Infection…","""2022-01-21""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc""","""public and global health""","""https://www.medrxiv.org/conten…","""IntroductionCurrently, informa…",null,"""medrxiv"""
"""10.1101/2023.02.05.23285484""","""The Hidden Footprints of Plate…","[""Oguz, A. K."", "" Oygur, C. S."", … "" Ergun, I.""]","""Ali Kemal Oguz""","""Baskent University School of M…","""2023-02-07""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""rheumatology""","""https://www.medrxiv.org/conten…","""Platelets have grabbed great a…",null,"""medrxiv"""
"""10.1101/2021.05.27.21257843""","""Health expenditure among the o…","[""shrestha, R."", "" Shakya, A.""]","""Rasmita shrestha""","""B.P. Koirala Institute of Heal…","""2021-06-01""",1,"""PUBLISHAHEADOFPRINT""","""cc_no""","""health economics""","""https://www.medrxiv.org/conten…","""IntroductionOut of Pocket (OOP…","""10.1155/2024/5571393""","""medrxiv"""


In [37]:
(
    df_cleaned
    # .group_by("id").agg(
    #     pl.col("version").sort_by("version")
    # )
    .with_columns(
        pl.col("published").alias("curated_doi"),
        pl.col("category").alias("primary_category"),
        pl.col("date").alias("published_date"),
    )
    .select([
        # pl.col("id"),
        pl.col("doi"),
        # pl.col("categories"),
        pl.col("primary_category"),
        pl.col("curated_doi"),
        pl.col("published_date"),
        # pl.col("last_updated_date"),
        pl.col("server"),
    ]).write_ndjson(f"{data_path}/medrxiv/medrxiv_preprints.dedup.jsonl")
)

# cleaning test

In [31]:
df_unique.filter(
    pl.col("published").is_not_null(),
    pl.col("published")!="",
    pl.col("published").is_duplicated()
).sort("published")

doi,title,authors,author_corresponding,author_corresponding_institution,date,version,type,license,category,jatsxml,abstract,published,server
str,str,list[str],str,str,str,i64,str,str,str,str,str,str,str
"""10.1101/2020.10.30.20223420""","""Racial and Ethnic Disparities …","[""Caraballo, C."", "" Massey, D."", … "" Krumholz, H.""]","""Harlan Krumholz""","""Yale University""","""2020-11-04""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""health systems and quality imp…","""https://www.medrxiv.org/conten…","""ImportanceRacial and ethnic di…","""10.1001/jama.2021.9907""","""medrxiv"""
"""10.1101/2020.10.30.20223487""","""Racial and Ethnic Disparities …","[""Mahajan, S."", "" Caraballo, C."", … "" Krumholz, H. M.""]","""Harlan M Krumholz""","""Yale School of Medicine""","""2020-11-04""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""health policy""","""https://www.medrxiv.org/conten…","""ImportanceThirty-five years ag…","""10.1001/jama.2021.9907""","""medrxiv"""
"""10.1101/2024.07.02.24309869""","""Posttraumatic stress-related w…","[""Quide, Y."", "" Kamarul, K."", "" Gustin, S. M.""]","""Yann Quidé""","""UNSW Sydney""","""2024-07-03""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""psychiatry and clinical psycho…","""https://www.medrxiv.org/conten…","""BackgroundPosttraumatic stress…","""10.1002/ejp.70034""","""medrxiv"""
"""10.1101/2024.05.09.24307012""","""Posttraumatic stress symptoms …","[""Quide, Y."", "" Hesam-Shariati, N."", … "" Gustin, S. M.""]","""Yann Quidé""","""NeuroRecovery Research Hub, Sc…","""2024-05-09""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""pain medicine""","""https://www.medrxiv.org/conten…","""BackgroundPosttraumatic stress…","""10.1002/ejp.70034""","""medrxiv"""
"""10.1101/2023.03.26.23287768""","""Functional Magnetic Resonance …","[""Foster, S. L."", "" Breukelaar, I. A."", … "" Korgaonkar, M. S.""]","""Sheryl L Foster""","""The University of Sydney""","""2023-03-27""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""radiology and imaging""","""https://www.medrxiv.org/conten…","""The amygdalae are a pair of sm…","""10.1002/jmri.28836""","""medrxiv"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""10.1101/2021.08.23.21262489""","""The COVID-19 Pandemic and Ment…","[""Zhang, S."", "" Sun, L."", … "" Li, D.""]","""Dongmei Li""","""University of Rochester Medica…","""2021-08-30""",1,"""PUBLISHAHEADOFPRINT""","""cc_no""","""infectious diseases""","""https://www.medrxiv.org/conten…","""BackgroundMental health illnes…","""10.34133/2022/9758408""","""medrxiv"""
"""10.1101/2020.05.04.20091017""","""Effects of home confinement on…","[""AMMAR, A."", "" Trabelsi, K."", … "" Sc""]","""ACHRAF AMMAR""","""Otto-Von-Guericke University""","""2020-05-08""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""public and global health""","""https://www.medrxiv.org/conten…","""BackgroundAlthough recognised …","""10.5114/biolsport.2020.96857""","""medrxiv"""
"""10.1101/2020.05.05.20091066""","""Social participation and life …","[""AMMAR, A."", "" Chtourou, H."", … "" S""]","""ACHRAF AMMAR""","""Otto-Von-Guericke University""","""2020-05-09""",1,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""public and global health""","""https://www.medrxiv.org/conten…","""BackgroundPublic health recomm…","""10.5114/biolsport.2020.96857""","""medrxiv"""


In [29]:
df_cleaned.group_by("author_corresponding").agg("date").filter(pl.col("date").list.len()>1)

author_corresponding,date
str,list[str]
"""Suresh K. Bhavnani""","[""2022-02-28"", ""2023-01-28""]"
"""Agnes Floeel""","[""2024-09-06"", ""2022-04-04""]"
"""Madhumita Shrotri""","[""2021-05-15"", ""2020-08-29""]"
"""Amilcar Tanuri""","[""2021-07-06"", ""2024-04-26""]"
"""Kalyan Das""","[""2020-06-12"", ""2020-06-12""]"
…,…
"""Paula Meireles""","[""2021-10-18"", ""2021-03-02""]"
"""Suruchi Mishra""","[""2021-12-29"", ""2023-08-28""]"
"""Brittany J Johnson""","[""2022-01-25"", ""2020-12-19""]"


In [36]:
pl.Config.set_fmt_str_lengths(n=300)
df_cleaned.filter(pl.col("version")>1)

doi,title,authors,author_corresponding,author_corresponding_institution,date,version,type,license,category,jatsxml,abstract,published,server
str,str,list[str],str,str,str,i64,str,str,str,str,str,str,str
"""10.1101/2020.08.04.20164061""","""Is Higher Viral Load in SARS-CoV-2 Associated With Death?""","[""Faico-Filho, K. S."", "" Passarelli, V. C."", "" Bellei, N.""]","""Victor Cabelho Passarelli""","""Federal University of Sao Paulo""","""2020-08-06""",2,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""infectious diseases""","""https://www.medrxiv.org/content/early/2020/08/06/2020.08.04.20164061.source.xml""","""BackgroundThere is no proven prognostic marker or adequate number of studies in patients hospitalized for coronavirus disease 2019 (COVID-19). MethodsWe conducted a retrospective cohort study of patients hospitalized with COVID-19 from March 14 to June 17, 2020, at Sao Paulo Hospital. SARS-CoV-2 vir…","""10.4269/ajtmh.20-0954""","""medrxiv"""
"""10.1101/2024.02.26.24303280""","""Identifying high-risk pre-term pregnancies using the fetal heart rate and machine learning""","[""Davis Jones, G."", "" Cooke, W."", "" Vatish, M.""]","""Gabriel Davis Jones""","""Nuffield Department of Women\'s and Reproductive Health, University of Oxford, United Kingdom""","""2024-02-27""",2,"""PUBLISHAHEADOFPRINT""","""cc_no""","""obstetrics and gynecology""","""https://www.medrxiv.org/content/early/2024/02/27/2024.02.26.24303280.source.xml""","""IntroductionFetal heart rate (FHR) monitoring is one of the commonest and most affordable tests performed during pregnancy worldwide. It is critical for evaluating the health status of the baby, providing real-time insights into the physiology of the fetus. While the relationship between patterns in…",null,"""medrxiv"""
"""10.1101/2020.02.19.20025387""","""Effectiveness of control strategies for Coronavirus Disease 2019: a SEIR dynamic modeling study""","[""Pan, J."", "" Yao, Y."", … "" Wang, W.""]","""Weibing Wang""","""Fudan University""","""2020-02-25""",2,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""infectious diseases""","""https://www.medrxiv.org/content/early/2020/02/25/2020.02.19.20025387.source.xml""","""BackgroundSince its first cases occurrence in Wuhan, China, the Coronavirus Disease 2019 (COVID-19) has been spreading rapidly to other provinces and neighboring countries. A series of intervention strategies have been implemented, but didnt stop its spread. MethodsTwo mathematical models have been …",null,"""medrxiv"""
"""10.1101/2023.10.25.23297418""","""Rapid DNA methylation-based classification of pediatric brain tumours from ultrasonic aspirate specimens""","[""Simon, M."", "" Kuschel, L. P."", … "" Euskirchen, P.""]","""Philipp Euskirchen""","""Charité Universitätsmedizin Berlin, Department of Neurology, Department of Neuropathology, Berlin, Germany, German Cancer Consortium (DKTK), Partner Site Berlin""","""2023-10-25""",2,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""oncology""","""https://www.medrxiv.org/content/early/2023/10/25/2023.10.25.23297418.source.xml""","""BackgroundAlthough cavitating ultrasonic aspirators are commonly used in neurosurgical procedures, the suitability of ultrasonic aspirator-derived tumor material for diagnostic procedures is still controversial. Here, we explore the feasibility of using ultrasonic aspirator-resected tumor tissue to …","""10.1007/s11060-024-04702-6""","""medrxiv"""
"""10.1101/2021.02.12.21251658""","""Emergence in late 2020 of multiple lineages of SARS-CoV-2 Spike protein variants affecting amino acid position 677""","[""Hodcroft, E. B."", "" Domman, D. B."", … "" Kamil, J. P.""]","""Jeremy P Kamil""","""Louisiana State University Health Sciences Center, Shreveport, Shreveport, LA, USA.""","""2021-02-21""",3,"""PUBLISHAHEADOFPRINT""","""cc_by_nc_nd""","""infectious diseases""","""https://www.medrxiv.org/content/early/2021/02/21/2021.02.12.21251658.source.xml""","""The severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2) spike pro

In [ ]:
df_cleaned.unique("").with_columns(
    pl.copl("doi").alias("doi"),
    pl.col("category").alias("primary_category"),
    pl.col("published").alias("curated_doi"),
    pl.col("date").alias("published_date"),
    pl.max("version").col("date")
).group_by("doi").max()

doi,title,authors,author_corresponding,author_corresponding_institution,date,version,type,license,category,jatsxml,abstract,published,server
str,str,list[str],str,str,str,i64,str,str,str,str,str,str,str
"""10.1101/19001024""","""Disease overarching mechanisms…","[""Bob Siegerink"", "" Joachim Weber"", … "" Tobias Pischon""]","""Tobias Pischon""","""Berlin Institute of Health (BI…","""2019-06-27""",1,"""""","""cc_by_nc""","""cardiovascular medicine""","""https://www.medrxiv.org/conten…","""BackgroundCardiovascular disea…",null,"""medrxiv"""
"""10.1101/19000109""","""Molecular profiling of neonata…","[""Daniel Costa"", "" Nuria Bonet"", … "" Robert Castelo""]","""Robert Castelo""","""Department of Experimental and…","""2019-06-25""",1,"""""","""cc_by""","""genetic and genomic medicine""","""https://www.medrxiv.org/conten…","""The fetal inflammatory respons…","""10.1111/febs.15578""","""medrxiv"""
"""10.1101/19000596""","""18F-AV1451 PET imaging and whi…","[""Nicolas Nicastro"", "" Patricia Vazquez Rodriguez"", … "" James B Rowe""]","""Nicolas Nicastro""","""University of Cambridge""","""2019-06-25""",1,"""""","""cc_by_nc_nd""","""neurology""","""https://www.medrxiv.org/conten…","""IntroductionProgressive supran…","""10.1007/s00415-019-09566-9""","""medrxiv"""
"""10.1101/19001115""","""Forecasting the impact of popu…","[""Chu-Chang Ku"", "" Peter J Dodd""]","""Chu-Chang Ku""","""School of Health and Related R…","""2019-06-28""",1,"""""","""cc_by""","""epidemiology""","""https://www.medrxiv.org/conten…","""BackgroundTuberculosis (TB) di…","""10.1371/journal.pone.0222937""","""medrxiv"""
"""10.1101/19000232""","""Optimising the use of molecula…","[""Munson, M."", ""Cresswell, B."", … ""Marks, M.""]","""Michael Marks""","""London School of Hygiene and T…","""2019-06-25""",1,"""""","""cc_by""","""infectious diseases""","""https://www.medrxiv.org/conten…","""IntroductionYaws is a neglecte…","""10.1093/trstmh/trz083""","""medrxiv"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""10.1101/19000174""","""Histopathologist Features Pred…","[""Myrtle J Van der Wel"", "" Helen G Coleman"", … "" Sybren L Meijer""]","""Marnix Jansen""","""UCL Cancer Institute""","""2019-06-25""",1,"""""","""cc_no""","""oncology""","""https://www.medrxiv.org/conten…","""ObjectiveGuidelines recommend …","""10.1136/gutjnl-2019-318985""","""medrxiv"""
"""10.1101/19000133""","""Prospective and External Evalu…","[""Nathan Brajer"", "" Brian Cozzi"", … "" Mark Sendak""]","""Nathan Brajer""","""Duke University School of Medi…","""2019-06-25""",1,"""""","""cc_by_nc_nd""","""health systems and quality imp…","""https://www.medrxiv.org/conten…","""The ability to accurately pred…","""10.1001/jamanetworkopen.2019.2…","""medrxiv"""
"""10.1101/19000521""","""Association of puberty timing …","[""Tuck Seng Cheng"", "" Felix R. Day"", … "" Ken K. Ong""]","""Tuck Seng Cheng""","""University of Cambridge""","""2019-06-28""",1,"""""","""cc_by_nc_nd""","""epidemiology""","""https://www.medrxiv.org/conten…","""OBJECTIVEWe aimed to systemati…","""10.1371/journal.pmed.1003017""","""medrxiv"""


In [38]:
import json

curated_doi = df_cleaned.filter(
    pl.col("published").is_not_null(),
    pl.col("published")!=""
).select("doi", "published")
curated_doi_dict = dict(zip(curated_doi["doi"], curated_doi["published"]))

unique_doi = df_cleaned.select("doi").to_series().unique().to_list()

print(len(curated_doi_dict), len(unique_doi))

with open(f"{base_path}/curated_medrxiv.json", "w") as f:
    json.dump(curated_doi_dict, f)
    
with open(f"{base_path}/unique_doi_medrxiv.json", "w") as f:
    json.dump(unique_doi, f)

32792 62019
